# Build a Supervised Model Table

**Goal:** Put everything for supervised models into supervised_model_table.csv with one row per issue-week with 155 rows.

In [1]:
from pathlib import Path
import pandas as pd

In [ ]:
# Unsupervised features table has the topic shares and shift scores
features = pd.read_csv("../DataFiles/tk_issue_week_features.csv")
# labels for repo has the suupervised labels
labels = pd.read_csv("../DataFiles/labels_for_repo.csv")
# answer key has the row id's issue week
answer_key = pd.read_csv("../DataFiles/answer_key.csv")
# volume summary has the original dataset tweet count
volume = pd.read_csv("../DataFiles/volume_summary.csv")
# noise floor has the expected week-to-week change from random variation
noise = pd.read_csv("../DataFiles/noise_floor.csv")

for name, table in [("features", features), ("labels", labels),
                    ("answer_key", answer_key), ("volume", volume), ("noise", noise)]:
    print(f"{name:12s} {table.shape}")

features     (155, 20)
labels       (150, 2)
answer_key   (150, 3)
volume       (155, 11)
noise        (155, 3)


### Merge labels with their respective issue and week. We removed these to prevent data leakage.

In [ ]:
# Make sure row id only shows up once since we the script saved results
# several times and we could rerun and duplicate results
merged_labels = labels.merge(answer_key, on="row_id", how="inner", validate="one_to_one")

assert len(merged_labels) == len(labels), "Some labels don't match answer key."
merged_labels = decoded_labels[["issue", "week", "label"]]
merged_labels.head()

,issue,week,label
0,abortion,25,0
1,foreign_policy,6,0
2,abortion,17,0
3,abortion,4,0
4,abortion,12,1


### Add labels to feature table

In [15]:
# Use left join to keep every row of feature table and add label where it exists
table = features.merge(decoded_labels, on=["issue", "week"], how="left", validate="one_to_one")

print(len(table))
print(f"Rows with a label: {table['label'].notna().sum()}")
print("Weeks with no label:", sorted(table.loc[table['label'].isna(), 'week'].unique()))

155
Rows with a label: 150
Weeks with no label: [np.int64(0)]


### Add original data volume. Will be used for ablation test.

In [10]:
volume_cols = ["issue", "week", "weekly_orig_total", "orig_tweet_count"]
table = table.merge(volume[volume_cols], on=["issue", "week"], how="left", validate="one_to_one")

assert table["orig_tweet_count"].notna().all(), "Some issue-weeks are missing a volume count."
table[["issue", "week", "n_tweets_issue", "orig_tweet_count"]].head()

,issue,week,n_tweets_issue,orig_tweet_count
0,abortion,0,121,5284
1,abortion,1,126,7731
2,abortion,2,64,1561
3,abortion,3,62,2230
4,abortion,4,66,962


### Add noise floor

In [11]:
table = table.merge(noise, on=["issue", "week"], how="left", validate="one_to_one")
print(f"Blank noise floors: {table['noise_floor'].isna().sum()}")
print(table.loc[table['noise_floor'].isna(), 'issue'].value_counts())

Blank noise floors: 16
issue
abortion    16
Name: count, dtype: int64


Final table should have one row per issue-week with 5 issues × 31 weeks = 155 rows.

In [ ]:
assert not table.duplicated(subset=["issue", "week"]).any(), "Duplicate issue-weeks"
assert len(table) == 155, f"Expected 155 rows, got {len(table)}"

# Sort in order by issue then week so previous row is the week before (for lags)
table = table.sort_values(["issue", "week"]).reset_index(drop=True)

table.to_csv("../DataFiles/supervised_model_table.csv", index=False)